## Lab 2 Warm-up : Inverse Kinematics on a toy robot

### Overview

ซ้อม flow ของ Lab 2 ทั้งหมดกับหุ่น **toy RRR** ที่รู้คำตอบแน่นอน ก่อนไปทำกับหุ่นของกลุ่มตัวเอง

| Section | ทำอะไร | ตรงกับ Lab 2 |
|---|---|---|
| 0.1 – 0.2 | โหลดหุ่น แล้ว cross-check MDH vs URDF | Part B Setup |
| A1 | แปลง `M_cam` จาก camera frame เป็น base frame ได้ `Pobj` | Part A ข้อ 1 |
| A2 | เขียนสมการ IK ให้ครบทุก branch | Part A ข้อ 2–4 |
| B1 – B4 | ส่งหุ่นไปที่ลูกบอล, เช็คด้วย FK, `ikine_LM`, ทุก branch | Part B ข้อ 1–4 |

เติมเฉพาะบรรทัดที่มี `# edit here` · เกณฑ์ผ่าน: error ≤ **0.03 m**

**หุ่น toy** (`toy_robot/robot.urdf`)

| | |
|---|---|
| joint 1 | revolute หมุนรอบแกน z ของ base · ไหล่สูงจาก base `d1 = 0.10` m |
| joint 2 | revolute ไหล่ ยกแขนขึ้น-ลง (`q2 > 0` = ยกขึ้น) |
| joint 3 | revolute ข้อศอก ห่างจากไหล่ `L2 = 0.15` m |
| tip | ห่างจากข้อศอก `L3 = 0.20` m |

ที่ `q = [0, 0, 0]` แขนเหยียดตรงตามแกน +x → tip อยู่ที่ `[0.35, 0, 0.10]`

Forward kinematics (แบบที่ทำใน LAB1) โดย `c1 = cos q1`, `s23 = sin(q2 + q3)`:

```
x = c1 (L2 c2 + L3 c23)
y = s1 (L2 c2 + L3 c23)
z = d1 + L2 s2 + L3 s23
```

### 0.1 Setup

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
if hasattr(Robot, 'URDF_read'):             # older roboticstoolbox
    links, name = Robot.URDF_read('toy_robot/robot.urdf', tld=HERE)[:2]
else:                                       # roboticstoolbox 1.5 and newer
    from roboticstoolbox.models.URDF import URDFRobot
    links, name = URDFRobot.URDF_read(HERE + '/toy_robot/robot.urdf')[:2]
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0, 0.0]   # home configuration
env.step()

In [ ]:
# Allowable syntax (reference only - do not run as-is):
# --- Part B : same list as Lab2_PartB.ipynb ---
# robot.fkine(q)          # forward kinematics -> transform T ; tip position is T.t
# np.linalg.norm(a - b)   # distance between two points
# SE3(*M)                 # make a target pose from a point M = [x, y, z]
# robot.ikine_LM(SE3(*M), q0=..., mask=[1,1,1,0,0,0], joint_limits=True)   # numerical IK -> sol
# sol.q                   # the joint solution it found
# sol.success             # did it converge (True/False)
# --- A1, A2 : warm-up only (in Lab 2 you do Part A on paper) ---
# np.linalg.inv(T)        # inverse of a 4x4 transform
# np.append(v, 1)         # [x, y, z] -> [x, y, z, 1]
# v[:3]                   # first three entries
# np.arctan2(y, x)  np.arccos(c)  np.sqrt(v)  np.cos(a)  np.sin(a)  np.pi

### 0.2 Cross-check your MDH model vs your URDF model

ตาราง MDH ของหุ่น toy ให้มาแล้ว รันสองเซลล์ล่างต้องได้ `MDH vs URDF: OK`

ใน Lab 2 จริงต้องเติมตาราง MDH ของหุ่นกลุ่มตัวเองจาก LAB1 ตรงนี้ · ลองแก้ `a=0.15` ของ joint 3 เป็น `0.16` แล้วรันใหม่ ดูว่า cross-check จับได้ (เสร็จแล้วแก้กลับ)

In [ ]:
# Syntax (reference only):
# RevoluteMDH(a=..., alpha=..., d=..., offset=...)        # revolute joint; joint variable is the angle
# PrismaticMDH(theta=..., a=..., alpha=..., offset=...)   # prismatic joint; joint variable is the distance
# robot_mdh.tool = SE3(x, y, z)                           # end-effector offset from the last MDH frame
from roboticstoolbox import DHRobot, RevoluteMDH, PrismaticMDH

# toy robot MDH table (given) - in Lab 2 you put your LAB1 table here
robot_mdh = DHRobot([
    RevoluteMDH(a=0.0,  alpha=0.0,       d=0.10),   # joint 1
    RevoluteMDH(a=0.0,  alpha=np.pi / 2, d=0.0),    # joint 2
    RevoluteMDH(a=0.15, alpha=0.0,       d=0.0),    # joint 3
], name='toy_mdh')

robot_mdh.tool = SE3(0.20, 0.0, 0.0)          # tip is L3 = 0.20 m along x of the last frame
print(robot_mdh)

In [ ]:
# Cross-check (nothing to edit): at the same q both models must put the tip at the same point
lo, hi = robot.qlim
q_tests = [np.zeros(robot.n)] + [lo + (hi - lo) * np.roll([0.25, 0.6, 0.85], k)[:robot.n] for k in range(3)]

worst = 0.0
for q in q_tests:
    p_mdh, p_urdf = robot_mdh.fkine(q).t, robot.fkine(q).t
    diff = float(np.linalg.norm(p_mdh - p_urdf))
    worst = max(worst, diff)
    print('q        :', np.round(q, 4))
    print('MDH  tip :', np.round(p_mdh, 4))
    print('URDF tip :', np.round(p_urdf, 4))
    print('pos diff :', round(diff, 4), 'm')
    print('-----------------------------------')

if worst <= 0.005:
    print('MDH vs URDF: OK')
else:
    print('MDH vs URDF: NOT MATCHING - fix your MDH table or your URDF, then run 0.1 and 0.2 again')

### A1. Camera frame → base frame (Part A ข้อ 1)

กล้องติดอยู่เหนือหุ่นและมองลงพื้น รายงานตำแหน่งชิ้นงาน `M_cam` ใน **camera frame** และโจทย์ให้ `T` = homogeneous transform ของ **base frame เทียบกับ camera frame**

ต้องการตำแหน่งชิ้นงานใน **base frame** (`Pobj`) จึงต้องใช้ **inverse ของ `T`** คูณกับ `[Mx, My, Mz, 1]`

1. ทำบนกระดาษก่อน: inverse ของ homogeneous transform มี rotation `Rᵀ` และ translation `−Rᵀp` แล้วเติมผลลง `hand_calc`
2. เขียนโค้ดหา `Pobj` แล้วดูว่า `diff` ใกล้ 0

> ใน Lab 2 จริง `M` และ `T` ของแต่ละกลุ่มอยู่ใน `Lab2_M_T_Handout.md` ค่า `Pobj` ที่ได้คือสิ่งที่ต้องเติมลงตัวแปร `M` ใน Part B

In [ ]:
M_cam = np.array([0.10, -0.05, 0.70])        # object position seen by the camera (camera frame) [m]
T = np.array([                               # base frame w.r.t. camera frame
    [1.0,  0.0,  0.0, -0.10],
    [0.0, -1.0,  0.0,  0.05],
    [0.0,  0.0, -1.0,  0.90],
    [0.0,  0.0,  0.0,  1.00],
])

Pobj = None                                  # edit here: object position in the base frame, [x, y, z]
hand_calc = np.array([0.0, 0.0, 0.0])        # edit here: your answer on paper

if Pobj is None:
    print('fill in Pobj')
else:
    print('Pobj     :', np.round(Pobj, 4))
    print('hand calc:', np.round(hand_calc, 4))
    print('diff     :', round(float(np.linalg.norm(Pobj - hand_calc)), 4), 'm')

### A2. IK equations, every branch (Part A ข้อ 2–4)

เขียน IK ของหุ่น toy จาก FK ด้านบน แล้วให้โค้ดคำนวณ **ทุก branch** ที่ไปถึง `Pobj`

- **q1** จาก `x = c1·r` และ `y = s1·r` · branch ที่สองคือหันไปอีกฝั่ง (`q1 ± π`) แล้วเอื้อมข้ามหัว
- **r** ระยะเอื้อมในแนวนอน *ตามทิศของแขน* (ฉาย `(x, y)` ลงทิศ `q1`) ติดลบได้
- **c3** `(r, s)` คือ tip ในระนาบของแขน จึงเหมือนแขน 2-link ในระนาบ ใช้กฎของ cos
- **q3** เครื่องหมาย `±` ของ `arccos` คือศอกงอขึ้นหรืองอลง
- **q2** มุมจากไหล่ไปหาเป้า ลบด้วยมุมที่ข้อศอกเพิ่มเข้ามา

หุ่น toy มีได้ `2 (ไหล่) × 2 (ศอก) = 4` branch · `wrap()` ดึงมุมกลับมาอยู่ใน `[-π, π)` ให้อยู่ใน joint limit

In [ ]:
d1, L2, L3 = 0.10, 0.15, 0.20


def wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi    # angle -> [-pi, pi)


def ik_toy(P, shoulder, elbow):
    x, y, z = P
    q1 = None    # edit here: base angle (shoulder = +1 faces the target, -1 faces away)
    r = None     # edit here: horizontal reach along the arm direction (can be negative)
    s = z - d1   # height of the target above the shoulder
    c3 = None    # edit here: cos(q3) from the law of cosines
    q3 = None    # edit here: elbow = +1 / -1 picks the sign
    q2 = None    # edit here
    if None in (q1, r, c3, q3, q2):
        return None
    return np.array([wrap(q1), wrap(q2), q3])


if Pobj is None:
    print('finish A1 first')
else:
    branches = [ik_toy(Pobj, sh, el) for sh in (+1, -1) for el in (+1, -1)]
    if branches[0] is None:
        print('fill in every "edit here" line in ik_toy')
    else:
        for i, qb in enumerate(branches):
            print('branch', i + 1, np.round(qb, 4))

### B1. target M and q (Part B ข้อ 1)

ใน Lab 2 ตัวแปร `M` คือเป้าหมายใน **base frame** (= `Pobj` จาก A1) ไม่ใช่ `M_cam` ที่กล้องให้มา

เลือก `q_goal` มา 1 branch แล้วดูว่าปลายมือไปแตะลูกบอลแดง · รันเซลล์ที่สองซ้ำกับ branch อื่นได้

In [ ]:
M = Pobj    # Lab 2 Part B: type the numbers of your Pobj here

marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))
marker.T = SE3(*M)
env.add(marker)
env.step()
print('target shown at', np.round(M, 4))

In [ ]:
# move the robot to q_goal  (tip should reach the ball)
q_goal = branches[0]    # try branches[1], [2], [3] too
robot.q = q_goal
env.step()

### B2. cross-check with FK (Part B ข้อ 2)

หา `tip` จาก FK ของ `q` แล้วหา `dist` = ระยะระหว่าง `tip` กับ `M` ต้องไม่เกิน 0.03 m

In [ ]:
q = q_goal

tip = None    # edit here
dist = None   # edit here

if dist is None:
    print('fill in tip and dist above')
else:
    print('tip:', np.round(tip, 4), '| M:', np.round(M, 4), '| err:', round(float(dist), 4))

### B3. let the toolbox solve IK (`ikine_LM`), compare with your branches (Part B ข้อ 3)

`ikine_LM` ให้คำตอบแค่ 1 branch ขึ้นกับจุดเริ่ม `q0` · ลองเปลี่ยน `q0` ให้ toolbox ไปเจอ branch อื่นอย่างน้อยอีก 1 branch

In [ ]:
sol = None    # edit here  (see Allowable syntax)

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('toolbox q:', np.round(sol.q, 4), '| ok', sol.success)
    print('hand q   :', np.round(q_goal, 4))
    near = int(np.argmin([np.linalg.norm(wrap(sol.q - qb)) for qb in branches]))
    print('closest to your branch', near + 1, np.round(branches[near], 4))

### B4. check every branch (Part B ข้อ 4)

ทุก branch ต้องได้ err ≤ 0.03 m และอยู่ใน joint limit · เซลล์นี้พาหุ่นไปทีละ branch ใน Swift ด้วย

ใน Lab 2 ต้องพิมพ์ค่าทุก branch จาก Part A ข้อ 4 ลงใน list `branches` เอง

In [ ]:
import time

lo, hi = robot.qlim
for i, qb in enumerate(branches):
    err = np.linalg.norm(robot.fkine(qb).t - M)
    inside = bool(np.all((qb >= lo) & (qb <= hi)))
    print('branch', i + 1, np.round(qb, 4), '-> err', round(float(err), 4), '| in limits', inside)
    robot.q = qb
    env.step()
    time.sleep(1.5)

### Next: your own robot

1. วางหุ่นของกลุ่มจาก LAB1 ลง `my_robot/` (ดู README)
2. รัน `python check_robot.py` ต้องได้ `[ok] ready for Lab 2`
3. คิด IK ของหุ่นตัวเองบนกระดาษ: แต่ละ joint เป็น R หรือ P, มีได้กี่ branch, branch ไหนติด joint limit